# 從零開始建立 Agent


**說明**

本教材參考 Udacity 的 AI Agents with LangChain and LangGraph 課程。


## 0. 匯入必要的套件


In [ ]:
import os
from openai import OpenAI

## 1. 如何使用 OpenAI 用戶端與你的 API 金鑰


若要連接 OpenAI，請先將你的 API 金鑰設定為名為 `OPENAI_API_KEY` 的系統環境變數。

接著讀取該環境變數來建立 OpenAI 用戶端。
```python
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
```


In [ ]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [ ]:
system_prompt = "Act as Senior Python Programmer. You don't know anything about other programming language, so don't provide answers about languanges like like Java."
user_question = "What is the Java Virtual Machine?"

In [ ]:
response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_question},
        ],
        temperature=0.0,
    )
response.choices[0].message.content

## 2. 定義 Agent 類別


### 目標


你的任務是實作一個代理程式，它可以：

- 使用可設定的參數初始化，包含名稱、角色、指令與模型參數。
- 將使用者訊息送到語言模型，並確保回應符合指定的角色與指令。
- 將 AI 產生的回應以字串形式回傳。


### 步驟


- 建立一個建構子，允許自訂代理程式的名稱、角色與指令。
- 確保代理程式會與語言模型互動，並將使用者訊息連同系統指令一起傳入。
- 實作一個方法來處理訊息，並確保能正確取得回應。


### 注意事項


- 角色與指令應該用來引導代理程式產生回應時的行為。
- 模型參數（例如 `temperature`）應該可以被設定。
- 保持代理程式的設計簡單，同時提供結構化的回應。


### 建構子


首先，建立一個名為 `Agent` 的類別，並在 `__init__` 方法中加入以下參數：

- `name`（預設值：`"Agent"`）
- `role`（預設值：`"Personal Assistant"`）
- `instructions`（預設值：`"Help users with any question"`）
- `model`（預設值：`"gpt-4o-mini"`）
- `temperature`（預設值：`0.0`）

建議讓用戶端可以在你的代理程式內部被存取。


### 呼叫方法


大多數代理程式框架都會提供一個 `invoke()` 方法。為了相容性，我們也採用相同做法。這個方法應該：

- 接收一則訊息作為輸入；
- 使用指定的模型與 `temperature`，將訊息送到大型語言模型的 API；
- 使用系統角色與使用者輸入來格式化 API 請求；
- 回傳大型語言模型的回應。

請注意，你的系統提示必須納入角色與指令，否則代理程式不會依照你期待的方式運作。


In [ ]:
class Agent():
    """A simple AI Agent"""

    def __init__(
        self,
        name:str = "Agent", 
        role:str = "Personal Assistant",
        instructions:str = "Help users with any question",
        model:str = "gpt-4o-mini",
        temperature:float = 0.0,
    ):
        self.name = name
        self.role = role
        self.instructions = instructions
        self.model = model
        self.temperature = temperature

        self.client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        
    def invoke(self, message: str) -> str:
        response = self.client.chat.completions.create(
            model = self.model,
            temperature=self.temperature,
            messages = [
                {
                    "role": "system",
                    "content": f"You're an AI Agent, your role is {self.role}, " 
                               f"and you need to {self.instructions}",
                },
                {
                    "role": "user",
                    "content": message,
                }

            ]
        )
        return response.choices[0].message.content

## 3. 建立代理程式並執行


建立一些特定用途的代理程式，並呼叫它們。


In [ ]:
agent = Agent()
response = agent.invoke("What is the capital of France?")
print("Agent role:", agent.role)
print("Default Agent Response:", response)

In [ ]:
travel_agent = Agent(
    role="Travel Assistant", 
    instructions="Provide travel recommendations.", 
    temperature=0.7
)
travel_response = travel_agent.invoke("Where should I go for vacation in December?")
print("Agent role:", travel_agent.role)
print("Travel Agent Response:", travel_response)

In [ ]:
math_tutor = Agent(
    role="Math Tutor", 
    instructions="Help students solve math problems step-by-step."
)
math_response = math_tutor.invoke("How do I solve a quadratic equation?")
print("Agent role:", math_tutor.role)
print("Math Tutor Response:", math_response)

In [ ]:
storyteller_agent = Agent(
    role="Storyteller", 
    instructions="Create imaginative stories.", 
    temperature=0.9
)
story_response = storyteller_agent.invoke("Tell me a story about a dragon and a wizard.")
print("Agent role:", storyteller_agent.role)
print("Creative Agent Response:", story_response)

## 4. 進階實驗


現在你已經理解它的運作方式，可以嘗試做一些新的實驗。

- 嘗試不同的角色與指令。
- 調整 `temperature`，觀察它如何影響回應。
- 用真實世界的問題測試代理程式。
